In [32]:
# generates the plots of replication time and inter-origin distance, as stratified by a signature

In [33]:
import pickle
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [34]:
# load the result of RFD signature extraction
exp_name='non_clustered'
dataset = '_pancan_sel'
pkl_file = '../data/interim/sig_extraction/SV_SigExtraction_nmf_RFD_'+exp_name+dataset+'.pkl'
X_file = '../data/interim/sig_extraction/X_'+exp_name+dataset+'.csv'
with open(pkl_file, 'rb') as f:
    model_loaded = pickle.load(f)
    deNovoSigs = pd.DataFrame(model_loaded.W)
    H_sel = pd.DataFrame(model_loaded.H.T)
    sil_scores = model_loaded.sil_score
# load the catalogue
X = pd.read_csv(X_file, index_col=0)

deNovoSigs.columns = [f"Signature {i+1}" for i in range(deNovoSigs.shape[1])]
H_sel.columns = [f"Signature {i+1}" for i in range(deNovoSigs.shape[1])]
H_sel.index = model_loaded.samples
deNovoSigs.index = model_loaded.features
# order tge sugbature channels according to the catalogue
deNovoSigs_expanded = deNovoSigs.reindex(X.index)
deNovoSigs = deNovoSigs_expanded

In [35]:
plotFolder = '../data/processed/Figures/RFD_sig_assignments/'
os.makedirs(plotFolder, exist_ok=True)

In [36]:
# P(S|C) \prop P(C|S)P(S)
# multiply the signature matrix (P(C|S) by the exposures matrix P(S)
# choose max for each sample


In [37]:
# calcaulate probabilities that in a sample, a given SV channel was generated by a given signature
sample_channel_sig_assignments = []
for sam in model_loaded.samples:
    sam_exp = H_sel.loc[sam,:]
    sam_exp = sam_exp/sam_exp.sum()
    sam_exp_m = pd.DataFrame(np.tile(sam_exp.to_numpy(), (deNovoSigs.shape[0], 1)))
    sam_exp_m.index =     deNovoSigs.index
    sam_exp_m.columns=deNovoSigs.columns
    
    psig_by_channel = deNovoSigs * sam_exp_m
    highest_sig_per_channel = pd.DataFrame(psig_by_channel.idxmax(axis=1))
    highest_sig_per_channel = highest_sig_per_channel.reset_index()
    highest_sig_per_channel['sample'] = sam
    # add the data frame to a list
    sample_channel_sig_assignments.append(highest_sig_per_channel)

In [38]:
# concatenate to a pan-sample data frame
all_sample_channel_assignments = pd.concat(sample_channel_sig_assignments, ignore_index=True)
all_sample_channel_assignments.columns = ['label_RFD', 'sigID', 'sample']

In [39]:
all_sample_channel_assignments.head(2)

,label_RFD,sigID,sample
0,non-clustered_del_1-10Kb_L_L,Signature 10,8f558713-f32b-403b-aedf-c79efeb41c67
1,non-clustered_del_1-10Kb_L_R,Signature 11,8f558713-f32b-403b-aedf-c79efeb41c67


In [40]:
# load the SVs
# assign SVs to the new signature

In [41]:
# see svig/cataloguePCAWG notebook
pcawg_svs = pd.read_csv('~/projects/rsignatures/data/processed/pcawg.rearrs.annot.csv')

In [42]:
pcawg_svs['label_RFD'] = pcawg_svs['label'] + '_'+ pcawg_svs['rf_dir']

In [43]:
pcawg_svs.head(2)

,Unnamed: 0,chrom1,start1,end1,chrom2,start2,end2,sv_id,pe_support,strand1,strand2,svclass,svmethod,sample,length,id,is.clustered,bkdist,catalogue.label,label1,label2,label3,label,max.Ref.Sig,sv_id_global,repliseq_bp1,repliseq_bp2,repliseq_avg,repliseq_mp,rf_dir_bp1,rf_dir_bp2,rf_dir,expr_fpkm_bp1,expr_fpkm_bp2,expr_fpkm_avg,origin_density,label_RFD
0,SVMERGE16_51800588-c622-11e3-bf01-24c6515278c0,1,36887569,36887570,1,36892311,36892312,SVMERGE16,49,+,-,deletion,SNOWMAN_BRASS_dRANGER_DELLY,51800588-c622-11e3-bf01-24c6515278c0,4742.0,1,False,4742,non-clustered_del_1-10Kb,non-clustered,_del,_1-10Kb,non-clustered_del_1-10Kb,Ref.Sig.R5,SVMERGE16_51800588-c622-11e3-bf01-24c6515278c0,77.1880,77.0929,77.14045,77.1405,R,R,R_R,3.695367,3.695367,3.695367,0.000057,non-clustered_del_1-10Kb_R_R
1,SVMERGE28_51800588-c622-11e3-bf01-24c6515278c0,1,43007396,43007397,1,43021859,43021860,SVMERGE28,89,+,-,deletion,SNOWMAN_BRASS_DELLY,51800588-c622-11e3-bf01-24c6515278c0,14463.0,2,False,14463,non-clustered_del_10-100Kb,non-clustered,_del,_10-100Kb,non-clustered_del_10-100Kb,Ref.Sig.R5,SVMERGE28_51800588-c622-11e3-bf01-24c6515278c0,66.2956,66.3973,66.34645,66.3437,R,L,R_L,0.399655,0.399655,0.399655,0.000048,non-clustered_del_10-100Kb_R_L


In [44]:
# merge SVs with per-sample signature assignments
pcawg_svs_m = pd.merge(pcawg_svs, all_sample_channel_assignments, left_on=['sample', 'label_RFD'], right_on=['sample', 'label_RFD'])

In [45]:
# make a boxplot: middpoint replication timing by signature

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order + sorted(other_sigs)
full_order.reverse()

# To enforce order manually, better use seaborn:
# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

pcawg_svs_m['origin_density_perMb'] = pcawg_svs_m['origin_density'] * (10**6)

plt.figure(figsize=(5, 8))
sns.boxplot(
    data=pcawg_svs_m,
    y='sigID',
    x='origin_density_perMb',
    order=full_order,
    orient='h',
    color='lightblue'
)
plt.xlim(0,0.00011*(10**6))
plt.title("Repli-seq MP by Signature")
plt.xlabel("replication origins per Mb")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()
plt.savefig(plotFolder+"/Repliseq_by_Signature_all_perMb.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

In [46]:
# same as above, but only show selected signatures

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order
full_order.reverse()

pcawg_svs_m_sel = pcawg_svs_m[pcawg_svs_m['sigID'].isin(order)]

# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

# Create the figure
plt.figure(figsize=(5, 5))
sns.boxplot(
    data=pcawg_svs_m_sel,
    y='sigID',
    x='repliseq_mp',
    order=full_order,
    orient='h',
    color='lightblue'
)

plt.title("Repli-seq MP by Signature")
plt.xlabel("repliseq_mp")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()

# --- Save as vector PDF with editable text ---
plt.savefig(plotFolder + "Repliseq_by_Signature_sel.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

In [47]:
# origin density

# Define desired order for top-to-bottom display
order = ['Signature 4', 'Signature 3', 'Signature 2', 'Signature 1', 'Signature 9', 'Signature 8']

# Add remaining signatures that aren’t in the specified list
other_sigs = [s for s in pcawg_svs_m['sigID'].unique() if s not in order]
full_order = order
full_order.reverse()

pcawg_svs_m_sel = pcawg_svs_m[pcawg_svs_m['sigID'].isin(order)]

# --- Set PDF backend for vector output ---
plt.rcParams['pdf.fonttype'] = 42   # Ensures fonts are stored as text, not curves
plt.rcParams['ps.fonttype'] = 42    # Same for PostScript
plt.rcParams['svg.fonttype'] = 'none'  # For completeness (if exporting to SVG)

pcawg_svs_m_sel['origin_density_perMb'] = pcawg_svs_m_sel['origin_density'] * (10**6)

# Create the figure
plt.figure(figsize=(5, 5))
sns.boxplot(
    data=pcawg_svs_m_sel,
    y='sigID',
    x='origin_density_perMb',
    order=full_order,
    orient='h',
    color='lightblue'
)
plt.xlim(0,0.00011*(10**6))
plt.title("Repli-seq MP by Signature")
plt.xlabel("# repl origins per Mb")
plt.ylabel("Signature ID")
plt.gca().invert_yaxis()  # make plot start from top
plt.tight_layout()

plt.savefig(plotFolder + "Repli_origin_by_Signature_sel_perMb.pdf", format="pdf", bbox_inches="tight")

plt.close()
plt.show()

/tmp/ipykernel_68755/3788581570.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pcawg_svs_m_sel['origin_density_perMb'] = pcawg_svs_m_sel['origin_density'] * (10**6)


In [48]:
# check if the difference in distribution of origin density between the two signatures is significant
from scipy.stats import mannwhitneyu

v_sig1=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 1']['origin_density']
v_sig1=v_sig1[v_sig1.notna()]
v_sig9=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 9']['origin_density']
v_sig9=v_sig9[v_sig9.notna()]

stat, p = mannwhitneyu(v_sig1, v_sig9)

In [49]:
stat


np.float64(2641318.5)

In [50]:
p

np.float64(0.0009437975181354401)

In [51]:

v_sig8=pcawg_svs_m_sel[pcawg_svs_m_sel['sigID']=='Signature 8']['origin_density']
v_sig8=v_sig8[v_sig8.notna()]

stat, p = mannwhitneyu(v_sig8, v_sig9)

In [52]:
p

np.float64(2.3161899479194124e-31)